<a href="https://colab.research.google.com/github/muajnstu/Large_Scale_Implementation_of_DSK_Chain/blob/main/Downstram_Pipeline_of_Proposed_Method(CSR).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, f1_score)
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, roc_auc_score, recall_score, precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier
)

from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    Perceptron,
    SGDClassifier,
    PassiveAggressiveClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/muajnstu/Large_Scale_Implementation_of_DSK_Chain/refs/heads/main/filtered%20data/Saimon%20and%20Roni/Clustered_CSR_Data.csv')
X = df.drop(columns=['CSAT Score'])
y = df['CSAT Score']

X = df.drop(columns=['CSAT Score'])
y = df['CSAT Score']

In [ ]:
y_cat = pd.Categorical(y)
y_codes = y_cat.codes

**Applying Random over sampler(ros) on whole dataset**

In [ ]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
print("Class distribution after ros:", pd.Series(y_resampled).value_counts())

Class distribution after ros: CSAT Score
1    15888
4    15888
2    15888
0    15888
3    15888
7    15888
9    15888
8    15888
5    15888
6    15888
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=46, stratify=y_resampled
)

**Implementation**

In [ ]:
def print_metrics(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    num_classes = cm.shape[0]

    specificity = 0
    sensitivity = 0
    gmean = 0
    type1 = 0
    type2 = 0
    fmeasure = 0
    auc = 0

    if num_classes == 2:
        TN, FP, FN, TP = cm.ravel()
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        gmean = np.sqrt(specificity * sensitivity)
        type1 = FP / (FP + TN) if (FP + TN) > 0 else 0
        type2 = FN / (TP + FN) if (TP + FN) > 0 else 0
        fmeasure = f1_score(y_true, y_pred, pos_label=1)
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob[:, 1])
            except Exception:
                auc = 0
    else:
        TP = np.diag(cm)
        FP = np.sum(cm, axis=0) - TP
        FN = np.sum(cm, axis=1) - TP
        TN = np.sum(cm) - (FP + FN + TP)

        specificity = np.mean([
            TN[i] / (TN[i] + FP[i]) if (TN[i] + FP[i]) > 0 else 0 for i in range(num_classes)
        ])
        sensitivity = np.mean([
            TP[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        gmean = np.sqrt(specificity * sensitivity)
        type1 = np.mean([
            FP[i] / (FP[i] + TN[i]) if (FP[i] + TN[i]) > 0 else 0 for i in range(num_classes)
        ])
        type2 = np.mean([
            FN[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        fmeasure = f1_score(y_true, y_pred, average='macro')
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
            except Exception:
                auc = 0

    metrics_dict = {
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "G-Mean": gmean,
        "Type I Error": type1,
        "Type II Error": type2,
        "F1 Score": fmeasure,
        "AUROC": auc
    }
    return metrics_dict

In [ ]:
def run_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
    except Exception:
        y_prob = None
    print(f"\nModel: {name}")
    metrics = print_metrics(y_test, y_pred, y_prob)
    # Print metrics for display, as print_metrics no longer prints
    for key, value in metrics.items():
        print(f"{key:13}: {value:.4f}")
    return metrics

In [ ]:
ml_models = {
    # Original models
    "RandomForest": RandomForestClassifier(random_state=42),
    "ExtraTrees": ExtraTreesClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
    "RidgeClassifier": RidgeClassifier(random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "Perceptron": Perceptron(random_state=42),
    "SGDClassifier": SGDClassifier(random_state=42),
    #"KNN": KNeighborsClassifier(n_neighbors=3),
    "PassiveAggressive": PassiveAggressiveClassifier(random_state=42),
    #"RBFSVM": SVC(kernel='rbf', probability=True, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.01),
    "LightGBM": LGBMClassifier(verbosity=-1, random_state=42),


    "VotingSoft": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='soft',
        n_jobs=-1
    ),


    "VotingHard": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='hard',
        n_jobs=-1
    ),


    "VotingWeighted": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        voting='soft',
        weights=[2, 3, 2, 3, 3],
        n_jobs=-1
    ),


    # ── Stacking 1: Tree ensembles → Logistic Regression meta ──────────────
    "Stacking_LR": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        final_estimator=LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),
}

dividing the ml models for efficient running

In [ ]:
ml_models_part1 = {}
ml_models_part2 = {}

# Split ml_models into two parts
model_names = list(ml_models.keys())

for i, name in enumerate(model_names):
    if i < 10:
        ml_models_part1[name] = ml_models[name]
    else:
        ml_models_part2[name] = ml_models[name]

print(f"ml_models_part1 contains {len(ml_models_part1)} models: {list(ml_models_part1.keys())}")
print(f"ml_models_part2 contains {len(ml_models_part2)} models: {list(ml_models_part2.keys())}")

ml_models_part1 contains 10 models: ['RandomForest', 'ExtraTrees', 'Bagging', 'GradientBoosting', 'LogisticRegression', 'RidgeClassifier', 'DecisionTree', 'NaiveBayes', 'Perceptron', 'SGDClassifier']
ml_models_part2 contains 8 models: ['PassiveAggressive', 'LDA', 'QDA', 'LightGBM', 'VotingSoft', 'VotingHard', 'VotingWeighted', 'Stacking_LR']


In [ ]:
for name, model in ml_models_part1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: RandomForest
Accuracy     : 0.9131
Sensitivity  : 0.9131
Specificity  : 0.9903
G-Mean       : 0.9510
Type I Error : 0.0097
Type II Error: 0.0869
F1 Score     : 0.9120
AUROC        : 0.9923

Model: ExtraTrees
Accuracy     : 0.9257
Sensitivity  : 0.9257
Specificity  : 0.9917
G-Mean       : 0.9581
Type I Error : 0.0083
Type II Error: 0.0743
F1 Score     : 0.9248
AUROC        : 0.9895

Model: Bagging
Accuracy     : 0.8985
Sensitivity  : 0.8985
Specificity  : 0.9887
G-Mean       : 0.9425
Type I Error : 0.0113
Type II Error: 0.1015
F1 Score     : 0.8969
AUROC        : 0.9904

Model: GradientBoosting
Accuracy     : 0.6360
Sensitivity  : 0.6360
Specificity  : 0.9596
G-Mean       : 0.7812
Type I Error : 0.0404
Type II Error: 0.3640
F1 Score     : 0.6127
AUROC        : 0.9620

Model: LogisticRegression
Accuracy     : 0.5584
Sensitivity  : 0.5584
Specificity  : 0.9509
G-Mean       : 0.7287
Type I Error : 0.0491
Type II Error: 0.4416
F1 Score     : 0.5547
AUROC        : 0.9506

Model: Ridg

In [ ]:
ml_models_part2_sub1 = {}
ml_models_part2_sub2 = {}

model_names_part2 = list(ml_models_part2.keys())
num_models_part2 = len(model_names_part2)

half_point = num_models_part2 // 2

for i, name in enumerate(model_names_part2):
    if i < half_point:
        ml_models_part2_sub1[name] = ml_models_part2[name]
    else:
        ml_models_part2_sub2[name] = ml_models_part2[name]

print(f"ml_models_part2_sub1 contains {len(ml_models_part2_sub1)} models: {list(ml_models_part2_sub1.keys())}")
print(f"ml_models_part2_sub2 contains {len(ml_models_part2_sub2)} models: {list(ml_models_part2_sub2.keys())}")

ml_models_part2_sub1 contains 4 models: ['PassiveAggressive', 'LDA', 'QDA', 'LightGBM']
ml_models_part2_sub2 contains 4 models: ['VotingSoft', 'VotingHard', 'VotingWeighted', 'Stacking_LR']


## Execute second batch of models (Sub-part 1)

In [ ]:
for name, model in ml_models_part2_sub1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: PassiveAggressive
Accuracy     : 0.2656
Sensitivity  : 0.2657
Specificity  : 0.9184
G-Mean       : 0.4939
Type I Error : 0.0816
Type II Error: 0.7343
F1 Score     : 0.1516
AUROC        : 0.0000

Model: LDA
Accuracy     : 0.5584
Sensitivity  : 0.5584
Specificity  : 0.9509
G-Mean       : 0.7287
Type I Error : 0.0491
Type II Error: 0.4416
F1 Score     : 0.5554
AUROC        : 0.9469

Model: QDA
Accuracy     : 0.5510
Sensitivity  : 0.5510
Specificity  : 0.9501
G-Mean       : 0.7235
Type I Error : 0.0499
Type II Error: 0.4490
F1 Score     : 0.5482
AUROC        : 0.9427

Model: LightGBM
Accuracy     : 0.7619
Sensitivity  : 0.7619
Specificity  : 0.9735
G-Mean       : 0.8612
Type I Error : 0.0265
Type II Error: 0.2381
F1 Score     : 0.7531
AUROC        : 0.9813


**Reasoning**:
Since the first batch of models (`ml_models_part1`) has been successfully executed, the next logical step is to execute the remaining models in `ml_models_part2`.



In [ ]:
ml_models_part2_sub2_sub1 = {}
ml_models_part2_sub2_sub2 = {}

model_names_part2_sub2 = list(ml_models_part2_sub2.keys())
num_models_part2_sub2 = len(model_names_part2_sub2)

half_point_sub2 = num_models_part2_sub2 // 2

for i, name in enumerate(model_names_part2_sub2):
    if i < half_point_sub2:
        ml_models_part2_sub2_sub1[name] = ml_models_part2_sub2[name]
    else:
        ml_models_part2_sub2_sub2[name] = ml_models_part2_sub2[name]

print(f"ml_models_part2_sub2_sub1 contains {len(ml_models_part2_sub2_sub1)} models: {list(ml_models_part2_sub2_sub1.keys())}")
print(f"ml_models_part2_sub2_sub2 contains {len(ml_models_part2_sub2_sub2)} models: {list(ml_models_part2_sub2_sub2.keys())}")



ml_models_part2_sub2_sub1 contains 2 models: ['VotingSoft', 'VotingHard']
ml_models_part2_sub2_sub2 contains 2 models: ['VotingWeighted', 'Stacking_LR']


In [ ]:
for name, model in ml_models_part2_sub2_sub1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: VotingSoft
Accuracy     : 0.9093
Sensitivity  : 0.9093
Specificity  : 0.9899
G-Mean       : 0.9487
Type I Error : 0.0101
Type II Error: 0.0907
F1 Score     : 0.9079
AUROC        : 0.9961


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: VotingHard
Accuracy     : 0.8171
Sensitivity  : 0.8171
Specificity  : 0.9797
G-Mean       : 0.8947
Type I Error : 0.0203
Type II Error: 0.1829
F1 Score     : 0.8050
AUROC        : 0.0000


In [ ]:
name_vw = 'VotingWeighted'
model_vw = ml_models_part2_sub2_sub2[name_vw]
run_model(name_vw, model_vw, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: VotingWeighted
Accuracy     : 0.8844
Sensitivity  : 0.8844
Specificity  : 0.9872
G-Mean       : 0.9344
Type I Error : 0.0128
Type II Error: 0.1156
F1 Score     : 0.8820
AUROC        : 0.9948


{'Accuracy': 0.8844410876132931,
 'Sensitivity': np.float64(0.8844427448781713),
 'Specificity': np.float64(0.9871601256915579),
 'G-Mean': np.float64(0.9343910376287446),
 'Type I Error': np.float64(0.012839874308442096),
 'Type II Error': np.float64(0.11555725512182871),
 'F1 Score': 0.8819842385463709,
 'AUROC': np.float64(0.9947988562393979)}

In [ ]:
stacking_1 = {
    "Stacking_LR": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        final_estimator=LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}

for name, model in stacking_1.items():
    run_model(name, model, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: Stacking_LR
Accuracy     : 0.9437
Sensitivity  : 0.9437
Specificity  : 0.9937
G-Mean       : 0.9684
Type I Error : 0.0063
Type II Error: 0.0563
F1 Score     : 0.9435
AUROC        : 0.9972


In [ ]:
stacking_2 = {
    "Stacking_LGBM": StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('dt', DecisionTreeClassifier(random_state=42)), # Added DecisionTreeClassifier as a replacement
            ('nb', GaussianNB())
        ],
        final_estimator=LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}

for name, model in stacking_2.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model: Stacking_LGBM
Accuracy     : 0.8982
Sensitivity  : 0.8982
Specificity  : 0.9887
G-Mean       : 0.9423
Type I Error : 0.0113
Type II Error: 0.1018
F1 Score     : 0.8964
AUROC        : 0.9904


In [ ]:
stacking_3 = {
    "Stacking_GB": StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42)),
            ('sgd', SGDClassifier(random_state=42, loss='modified_huber')),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_3.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_GB
Accuracy     : 0.9519
Sensitivity  : 0.9519
Specificity  : 0.9947
G-Mean       : 0.9731
Type I Error : 0.0053
Type II Error: 0.0481
F1 Score     : 0.9518
AUROC        : 0.9968


In [ ]:
stacking_4 = {
    "Stacking_RF": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB()),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42))
        ],
        final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_4.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_RF
Accuracy     : 0.9491
Sensitivity  : 0.9491
Specificity  : 0.9943
G-Mean       : 0.9715
Type I Error : 0.0057
Type II Error: 0.0509
F1 Score     : 0.9490
AUROC        : 0.9965


In [ ]:
stacking_5 = {
    "Stacking_ET": StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=ExtraTreesClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_5.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_ET
Accuracy     : 0.7813
Sensitivity  : 0.7813
Specificity  : 0.9757
G-Mean       : 0.8731
Type I Error : 0.0243
Type II Error: 0.2187
F1 Score     : 0.7808
AUROC        : 0.9841


Manually saved the outputs